# 💡 Matplotlib Practice Exercise 2: Solutions & Walkthroughs

This notebook contains **15 scenario-based challenges** reflecting real-world data engineering, machine learning diagnostics, and publication visualization tasks:
- Machine Learning convergence curves and automated early stopping callouts
- Multi-class Confusion Matrix heatmaps with normalized recall percentages
- ROC curves with Area Under Curve (AUC) and Precision-Recall curves with $F_1$ iso-contours
- Feature importance rankings with tree-variance error bars
- Complex multi-panel analytical dashboards via `subplot_mosaic()`
- Non-linear classification decision boundaries with probability contours
- Time series volatility with rolling moving averages and Bollinger Bands
- Customer cohort retention heatmaps
- Big Data high-density Hexbin visualizations
- Polar radar charts for multi-attribute model profiling
- Classifier calibration reliability diagrams with Brier score
- 3D optimization surfaces with gradient descent optimization paths
- Memory-safe high-throughput batch plotting pipelines

---
*Tip: Complete solutions are available in `05_matplotlib_exercise_2_solutions.ipynb`.*


### Scenario 1: ML Convergence Dynamics with Early Stopping Annotation

Given training and validation loss values across 40 epochs:
```python
epochs = np.arange(1, 41)
train_loss = 1.9 * np.exp(-epochs / 8.0) + 0.15 + np.random.normal(0, 0.02, 40)
val_loss = 2.0 * np.exp(-epochs / 9.0) + 0.32 + 0.0012 * (epochs - 16)**2 + np.random.normal(0, 0.025, 40)
```
1. Plot Train and Validation losses on the same Axes.
2. Programmatically identify the epoch where validation loss reaches its minimum.
3. Draw a dashed green vertical line at this optimal epoch.
4. Use `ax.annotate()` with a curved arrow pointing directly to the minimum point, displaying the optimal epoch and loss value.
5. Shade the region after epoch 20 in light orange to highlight the overfitting regime.


In [1]:
epochs = np.arange(1, 41)
train_loss = 1.9 * np.exp(-epochs / 8.0) + 0.15 + np.random.normal(0, 0.02, 40)
val_loss = 2.0 * np.exp(-epochs / 9.0) + 0.32 + 0.0012 * (epochs - 16)**2 + np.random.normal(0, 0.025, 40)

fig, ax = plt.subplots(figsize=(9, 4.8), dpi=100)
ax.plot(epochs, train_loss, color='#2b5c8f', lw=2, label='Train Loss')
ax.plot(epochs, val_loss, color='#d95f02', lw=2, label='Validation Loss')

opt_idx = np.argmin(val_loss)
opt_epoch = epochs[opt_idx]
opt_val = val_loss[opt_idx]

ax.axvline(opt_epoch, color='#2ca02c', linestyle='--', lw=1.8, label=f'Optimal Epoch ({opt_epoch})')
ax.axvspan(20, 40, color='#d95f02', alpha=0.1, label='Overfitting Zone')

ax.annotate(
    f'Min Val Loss: {opt_val:.3f}\nEpoch {opt_epoch}',
    xy=(opt_epoch, opt_val),
    xytext=(opt_epoch + 5, opt_val + 0.35),
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5, connectionstyle='arc3,rad=-0.2'),
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#ffffcc', edgecolor='#cccc00', alpha=0.9),
    fontweight='bold', fontsize=9
)

ax.set_title("Training & Validation Loss with Automated Early Stopping Detection", fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 1 completed: Convergence curves with early stopping marker and overfitting band


### Scenario 2: Normalized Multi-Class Confusion Matrix Heatmap

Given a 4x4 Confusion Matrix representing predictions across 4 medical diagnosis categories:
```python
labels = ['Healthy', 'Mild', 'Moderate', 'Severe']
cm = np.array([
    [142,  12,   4,   2],
    [ 15,  98,  14,   3],
    [  5,  18,  82,  15],
    [  1,   4,  11,  64]
])
```
1. Calculate the row-normalized percentage matrix (Recall % per class).
2. Render the normalized matrix using `ax.imshow()` with `cmap='Blues'` and `vmin=0, vmax=100`.
3. Overlay text inside each cell showing both the normalized percentage and the raw count (e.g. `88.8%\n(142)`).
4. Set text color dynamically to white for dark cells (>50%) and black for light cells.
5. Add a colorbar and appropriate axis labels.


In [2]:
labels = ['Healthy', 'Mild', 'Moderate', 'Severe']
cm = np.array([
    [142,  12,   4,   2],
    [ 15,  98,  14,   3],
    [  5,  18,  82,  15],
    [  1,   4,  11,  64]
])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=100, interpolation='nearest')

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Predicted Label", fontweight='semibold')
ax.set_ylabel("True Ground Truth", fontweight='semibold')

for i in range(len(labels)):
    for j in range(len(labels)):
        col = 'white' if cm_norm[i, j] > 50 else 'black'
        ax.text(j, i, f"{cm_norm[i, j]:.1f}%\n({cm[i, j]})",
                ha='center', va='center', color=col, fontweight='bold', fontsize=9)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Recall Rate (%)")
ax.set_title("Multi-Class Diagnostic Confusion Matrix", fontweight='bold', pad=12)
plt.show()


# Scenario 2 completed: Confusion matrix heatmap with dynamic high-contrast text labels


### Scenario 3: ROC Curve with AUC Score & Random Chance Baseline

Given FPR and TPR evaluation coordinates for two competing models:
```python
fpr = np.linspace(0, 1, 100)
tpr_rf = 1.0 / (1.0 + np.exp(-7 * (fpr - 0.18)))
tpr_rf[0] = 0.0
auc_rf = 0.941

tpr_lr = 1.0 / (1.0 + np.exp(-4 * (fpr - 0.28)))
tpr_lr[0] = 0.0
auc_lr = 0.812
```
1. Plot both ROC curves with distinct colors and line thicknesses.
2. Fill the area under the top-performing Random Forest curve with light blue (`alpha=0.15`).
3. Plot the random guessing diagonal baseline $y = x$ with a dashed gray line.
4. Include AUC scores directly inside the legend labels.
5. Set limits to `[-0.02, 1.02]` and grid on.


In [3]:
fpr = np.linspace(0, 1, 100)
tpr_rf = 1.0 / (1.0 + np.exp(-7 * (fpr - 0.18)))
tpr_rf[0] = 0.0
auc_rf = 0.941

tpr_lr = 1.0 / (1.0 + np.exp(-4 * (fpr - 0.28)))
tpr_lr[0] = 0.0
auc_lr = 0.812

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
ax.plot(fpr, tpr_rf, color='#2b5c8f', lw=2.5, label=f'Random Forest (AUC = {auc_rf:.3f})')
ax.plot(fpr, tpr_lr, color='#d95f02', lw=2.0, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
ax.fill_between(fpr, 0, tpr_rf, color='#2b5c8f', alpha=0.15)
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Baseline (AUC = 0.500)')

ax.set_title("Receiver Operating Characteristic (ROC) Comparison", fontweight='bold')
ax.set_xlabel("False Positive Rate (1 - Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity / Recall)")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='lower right', frameon=True)
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 3 completed: Multi-model ROC curves with shaded AUC and chance baseline


### Scenario 4: Precision-Recall Curve with F1 Score Iso-Curves

In heavily imbalanced fraud detection, Precision-Recall curves are superior to ROC curves.
1. Plot a simulated Precision-Recall curve:
   `recall = np.linspace(0.01, 1.0, 100)`
   `precision = 1.0 - 0.65 * recall**2`
2. Overlay **$F_1$ Iso-Contours**: Curves where $F_1 = \frac{2 \cdot P \cdot R}{P + R} = c$.
   For each $f_1 \in [0.2, 0.4, 0.6, 0.8]$:
   Compute $P = \frac{f_1 \cdot R}{2R - f_1}$ for $R > f_1 / 2$.
   Plot dashed gray lines and label each contour with its $F_1$ value.
3. Label axes and set limits to $[0, 1.05]$.


In [4]:
recall = np.linspace(0.01, 1.0, 100)
precision = 1.0 - 0.65 * recall**2

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)

# Plot F1 iso-contours
for f1 in [0.2, 0.4, 0.6, 0.8]:
    r_vals = np.linspace(f1 / 2 + 0.01, 1.0, 100)
    p_vals = (f1 * r_vals) / (2 * r_vals - f1)
    ax.plot(r_vals, p_vals, color='gray', linestyle=':', lw=1.0)
    ax.text(r_vals[-1] + 0.01, p_vals[-1], f'F1={f1:.1f}', color='gray', fontsize=8, va='center')

ax.plot(recall, precision, color='#2b5c8f', lw=2.5, label='Fraud Detection Classifier')
ax.fill_between(recall, 0, precision, color='#2b5c8f', alpha=0.15)

ax.set_title("Precision-Recall Curve with F1 Iso-Contours", fontweight='bold')
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1.08)
ax.set_ylim(0, 1.08)
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 4 completed: Precision-recall curve with F1 score iso-contours


### Scenario 5: Sorted Feature Importance with Tree-Variance Error Bars

Given 7 feature importances and their standard deviations computed across 100 decision trees:
```python
features = ['Seniority', 'Credit Limit', 'Transactions', 'Debt Ratio', 'Age', 'Savings', 'Delinquency']
importances = np.array([0.08, 0.26, 0.19, 0.14, 0.06, 0.16, 0.11])
std_devs = np.array([0.015, 0.035, 0.025, 0.020, 0.012, 0.022, 0.018])
```
1. Sort features ascending by importance.
2. Create a horizontal bar chart (`ax.barh`) with `xerr=std_devs` and `capsize=4`.
3. Set error bar line color to red and bar face color to steel blue.
4. Annotate the exact mean value next to each bar.


In [5]:
features = np.array(['Seniority', 'Credit Limit', 'Transactions', 'Debt Ratio', 'Age', 'Savings', 'Delinquency'])
importances = np.array([0.08, 0.26, 0.19, 0.14, 0.06, 0.16, 0.11])
std_devs = np.array([0.015, 0.035, 0.025, 0.020, 0.012, 0.022, 0.018])

idx = np.argsort(importances)
fig, ax = plt.subplots(figsize=(8, 4.8), dpi=100)
bars = ax.barh(features[idx], importances[idx], xerr=std_devs[idx], color='#2b5c8f', edgecolor='black',
               height=0.55, capsize=4, error_kw={'ecolor': '#d95f02', 'elinewidth': 1.5})

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.045, bar.get_y() + bar.get_height()/2.0, f"{w:.2f}",
            va='center', ha='left', fontsize=9, fontweight='bold')

ax.set_xlim(0, 0.36)
ax.set_title("Random Forest Feature Importance with Ensemble Variance", fontweight='bold')
ax.set_xlabel("Mean Gini Decrease (MDI)")
ax.grid(axis='x', linestyle=':', alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 5 completed: Sorted horizontal feature importance with variance error bars


### Scenario 6: Asymmetric Multi-Panel Dashboard via `subplot_mosaic`

Build a 3-panel analytics dashboard using `plt.subplot_mosaic()` with layout:
```python
blueprint = [
    ["summary", "detail_top"],
    ["summary", "detail_bot"]
]
```
With `width_ratios=[2, 1]`:
- `"summary"` (Left): Scatter plot of 150 points ($x, y$) with linear regression trendline.
- `"detail_top"` (Top Right): Histogram of residuals ($y - \hat{y}$).
- `"detail_bot"` (Bottom Right): Line plot of rolling mean of residuals.
Use `layout='constrained'` and set clean titles on each panel.


In [6]:
blueprint = [
    ["summary", "detail_top"],
    ["summary", "detail_bot"]
]

rng = np.random.default_rng(42)
x_pts = np.linspace(1, 10, 100)
y_pts = 2.5 * x_pts + 5 + rng.normal(0, 3, 100)

slope, intercept = np.polyfit(x_pts, y_pts, 1)
y_pred = slope * x_pts + intercept
residuals = y_pts - y_pred

fig, axd = plt.subplot_mosaic(blueprint, figsize=(11, 5.5), layout='constrained', gridspec_kw={'width_ratios': [2, 1]})

# 1. Summary
axd['summary'].scatter(x_pts, y_pts, color='#2b5c8f', alpha=0.6, edgecolors='none', label='Observations')
axd['summary'].plot(x_pts, y_pred, color='red', lw=2, label=f'Fit: y = {slope:.2f}x + {intercept:.2f}')
axd['summary'].set_title("Primary Regression Fit", fontweight='bold')
axd['summary'].grid(True, linestyle=':', alpha=0.5)
axd['summary'].legend()

# 2. Detail Top (Hist)
axd['detail_top'].hist(residuals, bins=15, color='#e27d60', edgecolor='white')
axd['detail_top'].set_title("Residual Error Distribution", fontsize=10, fontweight='bold')
axd['detail_top'].grid(True, linestyle=':', alpha=0.5)

# 3. Detail Bot (Rolling residual)
rolling_res = np.convolve(residuals, np.ones(5)/5, mode='valid')
axd['detail_bot'].plot(rolling_res, color='#41b3a3', lw=1.8)
axd['detail_bot'].axhline(0, color='gray', linestyle='--')
axd['detail_bot'].set_title("Rolling Mean Residual", fontsize=10, fontweight='bold')
axd['detail_bot'].grid(True, linestyle=':', alpha=0.5)

fig.suptitle("Linear Model Diagnostics Dashboard (subplot_mosaic)", fontsize=13, fontweight='bold')
plt.show()


# Scenario 6 completed: Multi-panel regression diagnostics dashboard


### Scenario 7: Machine Learning Decision Boundary with Probability Contours

Simulate a 2D classification dataset with two concentric ring clusters (e.g. circle dataset):
1. Create a 2D meshgrid over $X_1, X_2 \\in [-2.5, 2.5]$.
2. Compute model probability: $P = \\sigma(1.5(X_1^2 + X_2^2 - 2.0))$.
3. Render filled contour confidence regions with `ax.contourf(..., cmap='RdBu_r', levels=20, alpha=0.6)`.
4. Draw the critical decision boundary $P = 0.5$ in solid black with `linewidth=2.5`.
5. Scatter sample points for Class 0 (inside ring) and Class 1 (outside ring).


In [7]:
rng = np.random.default_rng(2026)
x1_grid = np.linspace(-2.5, 2.5, 200)
x2_grid = np.linspace(-2.5, 2.5, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)

# Probability function: Inner circle is Class 0, Outer is Class 1
Z_logits = 1.5 * (X1**2 + X2**2 - 2.0)
P_class1 = 1.0 / (1.0 + np.exp(-Z_logits))

fig, ax = plt.subplots(figsize=(7.5, 6), dpi=100)
cf = ax.contourf(X1, X2, P_class1, levels=20, cmap='RdBu_r', alpha=0.6)
cs = ax.contour(X1, X2, P_class1, levels=[0.5], colors='black', linewidths=2.5)
ax.clabel(cs, fmt='P=0.5 (Boundary)', fontsize=9)

# Scatter points
r_in = rng.uniform(0, 1.2, 60)
theta_in = rng.uniform(0, 2*np.pi, 60)
ax.scatter(r_in * np.cos(theta_in), r_in * np.sin(theta_in), color='#2166ac', edgecolors='white', label='Class 0 (Core)')

r_out = rng.uniform(1.6, 2.4, 60)
theta_out = rng.uniform(0, 2*np.pi, 60)
ax.scatter(r_out * np.cos(theta_out), r_out * np.sin(theta_out), color='#b2182b', edgecolors='white', label='Class 1 (Perimeter)')

cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("Predicted Class 1 Probability")
ax.set_title("Non-Linear Radial Decision Boundary & Confidence Field", fontweight='bold')
ax.legend(loc='lower left')
plt.show()


# Scenario 7 completed: Non-linear decision boundary with filled probability field


### Scenario 8: Financial Time Series with Rolling Average & Bollinger Bands

Given 90 days of simulated stock prices:
```python
days = np.arange(1, 91)
prices = 100 + np.cumsum(np.random.normal(0.2, 1.5, 90))
```
1. Compute the 7-day rolling moving average $\mu_7$ and 7-day rolling standard deviation $\sigma_7$.
2. Plot the raw daily closing price with a thin gray line.
3. Plot the 7-day moving average in bold navy.
4. Shade the **Bollinger Band** ($\mu_7 \\pm 2\\sigma_7$) in light blue with `alpha=0.2`.
5. Annotate the maximum price point on the chart with a pointer arrow.


In [8]:
days = np.arange(1, 91)
prices = 100 + np.cumsum(np.random.normal(0.2, 1.5, 90))

# Compute rolling stats (window = 7)
w = 7
rolling_mean = np.convolve(prices, np.ones(w)/w, mode='valid')
rolling_std = np.array([np.std(prices[i:i+w]) for i in range(len(prices) - w + 1)])
rolling_days = days[w-1:]

fig, ax = plt.subplots(figsize=(9, 4.8), dpi=100)
ax.plot(days, prices, color='#888888', lw=1.2, label='Daily Close')
ax.plot(rolling_days, rolling_mean, color='#1f77b4', lw=2.2, label='7-Day Moving Avg (SMA)')
ax.fill_between(rolling_days, rolling_mean - 2*rolling_std, rolling_mean + 2*rolling_std,
                 color='#1f77b4', alpha=0.18, label=r'Bollinger Band ($\mu \pm 2\sigma$)')

max_idx = np.argmax(prices)
ax.annotate(f'Peak: ${prices[max_idx]:.2f}',
            xy=(days[max_idx], prices[max_idx]),
            xytext=(days[max_idx] - 12, prices[max_idx] + 4),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffffcc', edgecolor='gray'),
            fontweight='bold', fontsize=9)

ax.set_title("Asset Volatility Analysis: 7-Day SMA & Bollinger Bands", fontweight='bold')
ax.set_xlabel("Trading Day")
ax.set_ylabel("Price ($USD)")
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 8 completed: Time series with 7-day rolling mean, Bollinger bands, and peak annotation


### Scenario 9: Customer Cohort Retention Triangular Heatmap

Given a monthly customer retention cohort matrix (percentages over 5 months):
```python
cohorts = ['Jan', 'Feb', 'Mar', 'Apr', 'May']
retention = np.array([
    [100, 42, 34, 28, 25],
    [100, 45, 36, 30, np.nan],
    [100, 48, 39, np.nan, np.nan],
    [100, 50, np.nan, np.nan, np.nan],
    [100, np.nan, np.nan, np.nan, np.nan]
])
```
1. Render the matrix using `ax.imshow()` with `cmap='Greens'` and `vmin=0, vmax=100`.
2. Format invalid/unborn future months (`np.nan`) cleanly with white background.
3. Stamp the exact retention percentage `f\"{val:.0f}%\"` on all valid cells with appropriate contrasting text color.
4. Set X-axis tick labels to `['Month 0', 'Month 1', 'Month 2', 'Month 3', 'Month 4']` and Y-axis to cohort names.


In [9]:
cohorts = ['Jan', 'Feb', 'Mar', 'Apr', 'May']
retention = np.array([
    [100, 42, 34, 28, 25],
    [100, 45, 36, 30, np.nan],
    [100, 48, 39, np.nan, np.nan],
    [100, 50, np.nan, np.nan, np.nan],
    [100, np.nan, np.nan, np.nan, np.nan]
])

fig, ax = plt.subplots(figsize=(7, 5.5), dpi=100)
# Mask NaNs
masked_ret = np.ma.masked_invalid(retention)
im = ax.imshow(masked_ret, cmap='Greens', vmin=0, vmax=100, interpolation='nearest')

ax.set_xticks(np.arange(5))
ax.set_yticks(np.arange(5))
ax.set_xticklabels([f'Month {i}' for i in range(5)], fontweight='semibold')
ax.set_yticklabels(cohorts, fontweight='semibold')
ax.set_ylabel("Cohort Signup Month", fontweight='bold')

for i in range(5):
    for j in range(5):
        val = retention[i, j]
        if not np.isnan(val):
            col = 'white' if val > 55 else 'black'
            ax.text(j, i, f"{val:.0f}%", ha='center', va='center', color=col, fontweight='bold', fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Retention Rate (%)")
ax.set_title("Customer Cohort Retention Rate (%)", fontweight='bold', pad=12)
plt.show()


# Scenario 9 completed: Customer cohort retention matrix with masked NaNs and percentage labels


### Scenario 10: 2D Hexbin Density Visualization with Logarithmic Counts

When visualizing massive geospatial or telemetry datasets (50,000+ points), individual scatter dots cause total visual occlusion.
1. Generate 50,000 points from a correlated bivariate Gaussian distribution:
   `x = rng.normal(0, 1.5, 50000)`
   `y = 0.8 * x + rng.normal(0, 0.8, 50000)`
2. Render a hexagonal bin density plot using `ax.hexbin(x, y, gridsize=40, cmap='inferno', bins='log')`.
3. Attach a colorbar with label `'Log10(Point Count)'`.
4. Set title and remove chart junk.


In [10]:
rng = np.random.default_rng(42)
x_big = rng.normal(0, 1.5, 50000)
y_big = 0.8 * x_big + rng.normal(0, 0.8, 50000)

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=100)
hb = ax.hexbin(x_big, y_big, gridsize=40, cmap='inferno', bins='log', mincnt=1)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label(r"$\log_{10}(\text{Sample Density})$")

ax.set_title("Big Data Density Mapping: Hexbin Tessellation (50,000 pts)", fontweight='bold')
ax.set_xlabel("Latent Feature X")
ax.set_ylabel("Latent Feature Y")
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 10 completed: Log-scaled 2D hexbin density chart


### Scenario 11: Multi-Attribute Model Competency Radar / Spider Chart

Compare two AI models across 6 key performance dimensions on a polar coordinate projection:
```python
attributes = ['Speed', 'Accuracy', 'Safety', 'Memory', 'Context', 'Tool Use']
model_a = [90, 85, 70, 60, 95, 80]
model_b = [65, 92, 90, 85, 80, 88]
```
1. Set up a polar Axes with `projection='polar'`.
2. Compute equal radian angles around the $2\\pi$ circle and close the polygon loops.
3. Plot both models with distinct colored lines and translucent polygon fills (`alpha=0.2`).
4. Set theta offset to top ($\\pi / 2$) and direction clockwise ($-1$).
5. Add attribute labels and a legend.


In [11]:
attributes = ['Speed', 'Accuracy', 'Safety', 'Memory', 'Context', 'Tool Use']
model_a = [90, 85, 70, 60, 95, 80]
model_b = [65, 92, 90, 85, 80, 88]

n = len(attributes)
angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
angles += angles[:1]
vals_a = model_a + model_a[:1]
vals_b = model_b + model_b[:1]

fig, ax = plt.subplots(figsize=(6.5, 6), subplot_kw=dict(projection='polar'), dpi=100)
ax.plot(angles, vals_a, color='#2b5c8f', lw=2, label='Model Alpha')
ax.fill(angles, vals_a, color='#2b5c8f', alpha=0.2)

ax.plot(angles, vals_b, color='#d95f02', lw=2, label='Model Beta')
ax.fill(angles, vals_b, color='#d95f02', alpha=0.2)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(attributes, fontweight='bold', fontsize=9)
ax.set_ylim(0, 100)
ax.set_title("AI Model Capability Profiling (Polar Radar Chart)", fontweight='bold', pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.show()


# Scenario 11 completed: Multi-attribute polar radar comparison chart


### Scenario 12: Classifier Calibration Reliability Diagram with Brier Score

Evaluate whether predicted probabilities correspond to empirical reality:
```python
conf_bins = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
emp_freq = np.array([0.12, 0.18, 0.33, 0.38, 0.52, 0.58, 0.73, 0.79, 0.91])
```
1. Plot the empirical calibration curve with markers against confidence bins.
2. Plot the perfect calibration reference diagonal line $y = x$ with a dashed black style.
3. Calculate the Brier calibration error: $\\frac{1}{N}\\sum (\\text{emp} - \\text{conf})^2$.
4. Display the Brier error score inside the legend label.
5. Set bounds to $[0, 1]$ and format axes as percentages.


In [12]:
conf_bins = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
emp_freq = np.array([0.12, 0.18, 0.33, 0.38, 0.52, 0.58, 0.73, 0.79, 0.91])
brier_err = np.mean((emp_freq - conf_bins)**2)

fig, ax = plt.subplots(figsize=(7, 5), dpi=100)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect Calibration')
ax.plot(conf_bins, emp_freq, marker='o', color='#2b5c8f', lw=2, markersize=6,
        label=f'Classifier (MSE/Brier = {brier_err:.4f})')

ax.set_title("Probability Calibration Reliability Diagram", fontweight='bold')
ax.set_xlabel("Mean Predicted Confidence")
ax.set_ylabel("Empirical True Positive Frequency")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='upper left', frameon=True)
ax.spines[['top', 'right']].set_visible(False)
plt.show()


# Scenario 12 completed: Classifier calibration curve with computed Brier score


### Scenario 13: 3D Optimization Surface with Gradient Descent Path

Visualize optimization over a 2D scalar field:
1. Define a 2D grid over $\theta_1, \theta_2 \\in [-2, 2]$.
2. Compute loss: $J(\\theta_1, \\theta_2) = \\theta_1^2 + 2\\theta_2^2 + \\sin(3\\theta_1)$.
3. Render a 3D surface plot using `ax.plot_surface()` with `cmap='viridis'`, `alpha=0.8`, and antialiased shading.
4. Overlay a simulated 12-step gradient descent trajectory path in red with circular markers.
5. Set camera elevation to 35 degrees and azimuth to -60 degrees.


In [13]:
t1 = np.linspace(-2, 2, 80)
t2 = np.linspace(-2, 2, 80)
T1, T2 = np.meshgrid(t1, t2)
J = T1**2 + 2*T2**2 + np.sin(3*T1)

# Optimization path
steps = 12
p_t1 = np.linspace(1.8, -0.4, steps)
p_t2 = np.linspace(1.5, 0.0, steps)
p_j = p_t1**2 + 2*p_t2**2 + np.sin(3*p_t1)

fig = plt.figure(figsize=(9, 5.5), dpi=100)
ax = fig.add_subplot(projection='3d')
surf = ax.plot_surface(T1, T2, J, cmap='viridis', alpha=0.8, antialiased=True)
ax.plot(p_t1, p_t2, p_j, color='red', marker='o', lw=2.5, markersize=4, label='GD Trajectory')

ax.view_init(elev=35, azim=-60)
ax.set_title(r"3D Optimization Trajectory on $J(\theta_1, \theta_2)$", fontweight='bold')
ax.set_xlabel(r"$\theta_1$")
ax.set_ylabel(r"$\theta_2$")
ax.set_zlabel("Cost J")
ax.legend()
plt.show()


# Scenario 13 completed: 3D loss surface with optimization trajectory path


### Scenario 14: Dual Coupled Physics Axes with `secondary_xaxis`

In optics, electromagnetic radiation can be represented either by **Wavelength (nm)** or by **Photon Energy (eV)** via the equation:
$$E = \\frac{hc}{\\lambda} \\approx \\frac{1239.84}{\\lambda}$$
1. Plot an absorption spectrum curve over wavelength $\\lambda \\in [400, 900]$ nm.
2. Attach a secondary top X-axis using `ax.secondary_xaxis('top', functions=(forward, inverse))` that displays the corresponding Photon Energy in electron-volts (eV).
3. Label both X-axes with their respective units and styling.


In [14]:
wavelength_nm = np.linspace(400, 900, 150)
absorption = np.exp(-((wavelength_nm - 550) / 80)**2) + 0.3 * np.exp(-((wavelength_nm - 720) / 50)**2)

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=100)
ax.plot(wavelength_nm, absorption, color='#2b5c8f', lw=2.2)
ax.set_xlabel("Wavelength $\lambda$ (nm)", color='#2b5c8f', fontweight='bold')
ax.set_ylabel("Optical Absorption (a.u.)")

def nm_to_ev(nm):
    return 1239.84 / np.maximum(nm, 1e-9)

def ev_to_nm(ev):
    return 1239.84 / np.maximum(ev, 1e-9)

sec_ax = ax.secondary_xaxis('top', functions=(nm_to_ev, ev_to_nm))
sec_ax.set_xlabel("Photon Energy E (eV)", color='#d95f02', fontweight='bold')

ax.set_title("Optical Absorption Spectrum with Dual Coupled Spectral Axes", fontweight='bold', pad=15)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()


# Scenario 14 completed: Dual coupled spectral axes (wavelength nm vs energy eV)


### Scenario 15: Memory-Safe High-Throughput Batch Figure Generator

Write a production-grade Python function `batch_plot_pipeline(n_batches=10)` that:
1. Simulates processing 10 successive mini-batches of evaluation data.
2. In each iteration, creates a new Figure, plots batch loss, and saves it to disk as `batch_{i}.png`.
3. Guarantees **zero memory leaks** by explicitly calling `plt.close(fig)` inside a `try...finally` block.
4. Checks and asserts that active Matplotlib figures in memory after the loop is exactly 0.
5. Cleans up the temporary files and prints the total number of successfully exported plots.


In [15]:
import os

def batch_plot_pipeline(n_batches=10):
    exported_files = []
    print(f"Active figures before batch processing: {len(plt.get_fignums())}")
    
    for i in range(n_batches):
        fig, ax = plt.subplots(figsize=(4, 2.5), dpi=80)
        try:
            batch_loss = np.random.uniform(0.1, 0.5, 20)
            ax.plot(batch_loss, color='#2b5c8f')
            ax.set_title(f"Batch {i+1} Telemetry", fontsize=8)
            filename = f"temp_batch_{i+1}.png"
            fig.savefig(filename, bbox_inches='tight')
            exported_files.append(filename)
        finally:
            plt.close(fig) # Immediate memory cleanup!
            
    active_figs = len(plt.get_fignums())
    print(f"Active figures after batch loop: {active_figs} (Memory fully released!)")
    
    # Cleanup temporary files
    for fn in exported_files:
        if os.path.exists(fn):
            os.remove(fn)
            
    print(f"Successfully generated, verified, and cleaned {n_batches} batch reports.")

batch_plot_pipeline(n_batches=10)


Active figures before batch processing: 0
Active figures after batch loop: 0 (Memory fully released!)
Successfully generated, verified, and cleaned 10 batch reports.
